In [33]:
import random
filename = "/Users/ziqiangcui/Desktop/data/lightgcn_new/train.txt"
user_items = {}
max_len = 0
cnt = 0
with open(filename, 'r') as file:
    for line in file:
        parts = line.strip().split()
        user_id = int(parts[0])
        item_ids = list(map(int, parts[1:]))
        max_len = max(max_len, len(item_ids))

        if len(item_ids) > 150:
            cnt+=1
            print(user_id)
            item_ids = random.sample(item_ids, 150)
        
        user_items[user_id] = item_ids


4
9
14
16
17
18
21
22
25
32
34
35
41
43
44
47
52
57
58
61
72
89
91
92
116
117
122
130
135
136
138
145
147
148
149
150
156
160
162
165
168
172
174
180
186
191
192
194
197
198
201
203
215
222
223
224
228
234
237
241
244
254
260
263
267
270
271
277
283
292
294
299
300
301
302
305
307
309
313
318
320
325
326
328
330
332
336
337
342
345
348
351
354
365
367
385
389
391
397
401
402
408
410
411
414
423
424
425
428
437
441
444
450
452
453
456
460
465
473
474
475
476
480
481
493
498
508
515
517
519
523
527
530
532
535
540
542
545
548
549
550
555
557
562
565
569
586
587
590
600
603
620
623
628
630
636
638
645
646
650
654
659
668
672
675
676
677
686
691
694
695
697
698
701
709
711
712
713
715
719
720
726
730
734
736
743
745
748
751
752
756
764
769
773
776
779
780
787
790
795
797
799
800
801
807
815
816
821
823
838
839
845
849
853
854
857
868
876
880
888
889
890
896
898
903
910
918
921
923
926
927
928
933
934
936
947
948
952
954
956
962
969
970
972
974
976
980
983
995
998
1000
1003
1009
1014
1016
1

In [13]:
cnt

1686

In [29]:
import pandas as pd
path = "/Users/ziqiangcui/Desktop/data/"
movies = pd.read_csv(path + "generated/ml1m_extended_movie.csv")

movies['genres'] = movies['genres'].combine_first(movies['Genres'])  
# Convert the 'Date' column to datetime format
movies['release_date'] = pd.to_datetime(movies['release_date'])
# Convert the 'Date' column to year-month format
movies['release_date'] = movies['release_date'].dt.to_period('M')
movies['release_date'] = movies['release_date'].combine_first(movies['Year']) 
movies['director'] = movies['director'].fillna("unknown")
movies['actors'] = movies['actors'].fillna("unknown")
movies['overview'] = movies['overview'].fillna("unknown")
movies['writer'] = movies['writer'].fillna("unknown")

item_list = "/Users/ziqiangcui/Desktop/data/lightgcn_new/item_map.txt"
item_map = {}


with open(item_list, 'r') as file:
    for line in file:
        key, value = line.strip().split()
        item_map[key] = int(value)
movies["item_id"] = movies["MovieID"].apply(lambda x: item_map.get(str(x)))

movies = movies[["MovieID",'item_id', 'Title','Genres', 'director', 'actors']]

In [30]:

user_text_dic = {}

for index, row in movies.iterrows():
    item_id = row['item_id']
    name = row['Title']
    genres = row['Genres']
    director = row['director']
    actors = row['actors']
    
    text_kg = f'{name}: {{"genres": {genres}, "director": "{director}", "main actors": "{actors}"}}'
    
    user_text_dic[item_id] = text_kg


user_text = {}
max_len = 0
for user, item_list in user_items.items():
    max_len = max(max_len, len(item_list))
    text_list = [user_text_dic[item] for item in item_list]
    result_string = "; ".join(text_list)
    user_text[user] = result_string

In [42]:
import json

def prompt_generation(id, text):
   
    analysis = \
    f"""
    Assuming you're a film expert with access to a viewer's movie-watching history, where each entry is formatted as "movie_name: genres: xx, director: xx, main actors: xx)".
    {text}
    Please analyze and summarize this user's viewing preferences from the aspects of movie genres, directors, and actors. Your response should be a coherent and fluent paragraph, not exceeding 100 words.
    Here is a sample output:"This user has a strong preference for action-packed films, often combined with elements of sci-fi, drama, and adventure. They frequently watch movies directed by Steven Spielberg and Ridley Scott, indicating a taste for high-quality, visually compelling storytelling. The user also enjoys films featuring iconic actors such as Harrison Ford, Arnold Schwarzenegger, and Paul Newman. Their viewing history includes classics like "Star Wars," "Jurassic Park," and "The Terminator," showcasing a penchant for both timeless blockbusters and genre-defining cinema. Overall, their tastes lean towards thrilling, well-crafted narratives with strong directorial vision and memorable performances."
    """
    
    return analysis


question_dic = {}
for id, text in user_text.items():
    prompt = prompt_generation(id, text)        
    question_dic[id] = prompt


print("The number of requests is: ", len(list(question_dic.keys())))

with open('/Users/ziqiangcui/Desktop/data/ml-1m/user_question_dic_v1.json', 'w') as f:
    json.dump(question_dic, f)

The number of requests is:  6040
